In [1]:
import time
import psutil
import os
from datetime import datetime
import pandas as pd
import numpy as np
import csv

In [2]:
# Cell 2: Load Data with proper type conversion
print("\nLOADING DATA")
print("-"*40)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Load data
df = pd.read_csv("movies_cleaned.csv")

# Convert numeric columns, coercing errors to NaN
numeric_cols = ['budget', 'revenue', 'vote_average', 'vote_count', 'popularity', 'runtime']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with missing critical values
df = df.dropna(subset=['vote_average', 'budget', 'revenue'])

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory

# Store metrics for summary
load_time = execution_time
load_memory = memory_delta
load_end_memory = end_memory

print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Rows loaded: {len(df):,}")
print(f"Columns: {len(df.columns)}")


LOADING DATA
----------------------------------------

 BENCHMARKS:
Time: 9.07 seconds
Memory delta: 2265.64 MB
Memory total: 2399.41 MB
CPU: 0.0%
Rows loaded: 590,202
Columns: 166


In [3]:
# Cell 3: Analysis 1 - Genres by Country and Release Year
print("\n" + "="*80)
print("ANALYSIS 1: GENRES BY COUNTRY AND RELEASE YEAR")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Prepare data
task6 = df.copy()
initial_rows = len(task6)
task6['year'] = pd.to_datetime(task6['release_date'], errors='coerce').dt.year
task6 = task6.dropna(subset=['year'])
task6['year'] = task6['year'].astype(int)

# Group by
genreCountryYear = task6.groupby(['genres', 'production_countries', 'year']).agg({
    'id': 'count',
    'vote_average': 'mean',
    'revenue': 'mean'
}).round(2)

genreCountryYear.columns = ['movie_count', 'avg_rating', 'avg_revenue']
genreCountryYear = genreCountryYear.reset_index()
genreCountryYear = genreCountryYear.sort_values(['year', 'production_countries', 'genres'])

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(genreCountryYear)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis1_time = execution_time
analysis1_memory = memory_delta

print(f"Total groups: {rows_processed:,}")
print(genreCountryYear.head(7))
print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Groups created: {rows_processed:,}")
print(f"Groups/second: {rows_per_second:,.0f}")
print(f"Unique years: {genreCountryYear['year'].nunique()}")
print(f"Unique countries: {genreCountryYear['production_countries'].nunique()}")


ANALYSIS 1: GENRES BY COUNTRY AND RELEASE YEAR
Total groups: 209,339
                    genres production_countries  year  movie_count  \
80369          Documentary       United Kingdom  1888            2   
80639          Documentary        United States  1890            2   
80640          Documentary        United States  1891            1   
80641          Documentary        United States  1892            1   
78904          Documentary               Poland  1893            1   
83834   Documentary, Drama        United States  1893            1   
102821               Drama        United States  1893            1   

        avg_rating  avg_revenue  
80369         5.68          0.0  
80639         4.70          0.0  
80640         4.90          0.0  
80641         3.50          0.0  
78904         1.00          0.0  
83834         2.70          0.0  
102821        5.60          0.0  

 BENCHMARKS:
Time: 0.49 seconds
Memory delta: 174.17 MB
Memory total: 2589.41 MB
CPU: 0.0%
Group

In [4]:
# Cell 4: Analysis 2 - Genres by Decade with Budget Bracket
print("\n" + "="*80)
print("ANALYSIS 2: GENRES BY DECADE WITH BUDGET BRACKET")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Create budget brackets
task6['budget_bracket'] = pd.cut(
    task6['budget'],
    bins=[-1, 0, 1000000, 10000000, 50000000, 100000000, float('inf')],
    labels=['Zero', '<1M', '1M-10M', '10M-50M', '50M-100M', '>100M']
)

# Group by
genrDecadeBudget = task6.groupby(['genres', 'decade', 'budget_bracket'], observed=True).agg(
    movie_count=('id', 'count'),
    avg_rating=('vote_average', 'mean'),
    total_rev=('revenue', 'sum')
).round(2).reset_index()

genrDecadeBudget = genrDecadeBudget[(genrDecadeBudget['total_rev'] > 0) & (genrDecadeBudget['budget_bracket'] != 'Zero')]
genrDecadeBudget = genrDecadeBudget.sort_values(['decade', 'genres', 'budget_bracket'])

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(genrDecadeBudget)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis2_time = execution_time
analysis2_memory = memory_delta

print(f"Total groups: {rows_processed:,}")
print(genrDecadeBudget.head(7))
print(f"\nBENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Groups created: {rows_processed:,}")
print(f"Groups/second: {rows_per_second:,.0f}")
print(f"Unique decades: {genrDecadeBudget['decade'].nunique()}")
print(f"Budget brackets: {genrDecadeBudget['budget_bracket'].nunique()}")


ANALYSIS 2: GENRES BY DECADE WITH BUDGET BRACKET
Total groups: 6,681
                                          genres  decade budget_bracket  \
5846   Adventure, Drama, Action, Science Fiction    1910            <1M   
11706                              Comedy, Drama    1910            <1M   
15182                               Crime, Drama    1910            <1M   
16625                                Documentary    1910            <1M   
17889                                      Drama    1910            <1M   
20233                             Drama, History    1910            <1M   
20551                        Drama, History, War    1910            <1M   

       movie_count  avg_rating  total_rev  
5846             1        6.20    8000000  
11706            1        6.50    8000000  
15182            1        5.00    1800000  
16625            2        6.85          1  
17889            8        2.95    3840133  
20233            1        7.10    1750000  
20551            2   

In [5]:
# Cell 5: Analysis 3 - Genre Performance by Production Company
print("\n" + "="*80)
print("ANALYSIS 3: GENRE PERFORMANCE BY PRODUCTION COMPANY")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Prepare data
companies = df.copy()
companies = companies.rename(columns={'production_companies_clean': 'production_company'})
companies['production_company'] = companies['production_company'].str.strip('[]').str.replace("'", "").str.split(', ')
companies = companies.explode('production_company')
companies = companies[companies['production_company'].notna() & (companies['production_company'] != '')]

companies['genres'] = companies['genres'].str.split(', ')
companies = companies.explode('genres')
companies = companies[companies['genres'].notna() & (companies['genres'] != '')]

# Group by
companyGenrePerf = companies.groupby(['production_company', 'genres']).agg(
    movie_count=('id', 'count'),
    avg_rating=('vote_average', 'mean'),
    total_rev=('revenue', 'sum')
).round(2).reset_index()

companyGenrePerf = companyGenrePerf[(companyGenrePerf['total_rev'] > 0) & (companyGenrePerf['movie_count'] >= 2)]
companyGenrePerf = companyGenrePerf.sort_values(['production_company', 'movie_count'], ascending=[True, False])

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(companyGenrePerf)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis3_time = execution_time
analysis3_memory = memory_delta

print(f"Total combinations: {rows_processed:,}")
print(companyGenrePerf.head())
print(f"\nBENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Combinations: {rows_processed:,}")
print(f"Combinations/second: {rows_per_second:,.0f}")
print(f"Unique companies: {companyGenrePerf['production_company'].nunique():,}")
print(f"Unique genres: {companyGenrePerf['genres'].nunique()}")
print(f"Avg revenue per combo: ${companyGenrePerf['total_rev'].mean():,.0f}")


ANALYSIS 3: GENRE PERFORMANCE BY PRODUCTION COMPANY
Total combinations: 39,779
                                    production_company genres  movie_count  \
52   "A.N.P.A. (Agence Nationale de Promotion de lA...  Drama            2   
89   "Agence Nationale pour la Cohésion Sociale et ...  Drama            6   
112                                             "Alin"  Drama           11   
115                                             "Alin"  Music            2   
144                      "Anarchists Convention Films"  Drama            7   

     avg_rating  total_rev  
52         3.00    3314254  
89         6.35     502392  
112        6.12    1370693  
115        5.60    1370693  
144        5.84    5008141  

BENCHMARKS:
Time: 7.87 seconds
Memory delta: -371.88 MB
Memory total: 2243.02 MB
CPU: 0.0%
Combinations: 39,779
Combinations/second: 5,053
Unique companies: 12,973
Unique genres: 19
Avg revenue per combo: $171,677,444


In [6]:
# Cell 6: Analysis 4 - Multiple vs Single Genre Performance
print("\n" + "="*80)
print("ANALYSIS 4: MULTIPLE VS SINGLE GENRE PERFORMANCE")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Count genres
task6['num_genres'] = task6['genres'].str.split(', ').str.len()

# Group by
genreCountStats = task6.groupby('num_genres').agg(
    movie_count=('id', 'count'),
    avg_rev=('revenue', 'mean'),
    median_rev=('revenue', 'median'),
    total_rev=('revenue', 'sum'),
    avg_rating=('vote_average', 'mean'),
    median_rating=('vote_average', 'median')
).round(2).reset_index()

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(genreCountStats)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis4_time = execution_time
analysis4_memory = memory_delta

print(genreCountStats)
print(f"\n BENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Genre count groups: {rows_processed}")
print(f"Groups/second: {rows_per_second:,.0f}")
print(f"Genre range: 1-{genreCountStats['num_genres'].max()} genres")
single_genre = genreCountStats[genreCountStats['num_genres']==1]['movie_count'].values[0] if len(genreCountStats[genreCountStats['num_genres']==1]) > 0 else 0
multi_genre = genreCountStats[genreCountStats['num_genres']>=3]['movie_count'].sum() if len(genreCountStats) > 0 else 0
print(f"Movies with 1 genre: {single_genre:,}")
print(f"Movies with 3+ genres: {multi_genre:,}")
print("="*80)


ANALYSIS 4: MULTIPLE VS SINGLE GENRE PERFORMANCE
    num_genres  movie_count       avg_rev  median_rev     total_rev  \
0            1       321831  1.898190e+05         0.0   61089649229   
1            2       176596  1.083959e+06         0.0  191422858749   
2            3        71098  5.149793e+06         0.0  366140008901   
3            4        16389  8.552903e+06         0.0  140173529894   
4            5         3416  1.294481e+07         0.0   44219466142   
5            6          694  9.647596e+06         0.0    6695431971   
6            7          130  1.864562e+06         0.0     242393073   
7            8           30  1.011340e+06         0.0      30340197   
8            9           11  4.026640e+03         0.0         44293   
9           10            3  0.000000e+00         0.0             0   
10          11            3  2.666667e+08         0.0     800000000   
11          12            1  0.000000e+00         0.0             0   

    avg_rating  median_rat

In [7]:
# Cell 7: Analysis 5 - Movie Ratings by Language per Decade
print("\n" + "="*80)
print("ANALYSIS 5: MOVIE RATINGS BY LANGUAGE PER DECADE")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Group by
langStats = task6.groupby(["decade", "main_spoken_language_clean"]).agg(
    movie_count=('id', 'count'),
    avg_rating=('vote_average', 'mean'),
    median_rating=('vote_average', 'median'),
    rating_std=('vote_average', 'std')
).round(2).reset_index()

langStats = langStats[langStats['movie_count'] >= 5]
langStats = langStats.sort_values(["decade", "movie_count"], ascending=[True, False])

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(langStats)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis5_time = execution_time
analysis5_memory = memory_delta

print(f"Total groups: {rows_processed:,}")
print(langStats.head(20))
print(f"\nBENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Language-decade groups: {rows_processed:,}")
print(f"Groups/second: {rows_per_second:,.0f}")
print(f"Unique decades: {langStats['decade'].nunique()}")
print(f"Unique languages: {langStats['main_spoken_language_clean'].nunique()}")
print("="*80)


ANALYSIS 5: MOVIE RATINGS BY LANGUAGE PER DECADE
Total groups: 616
    decade main_spoken_language_clean  movie_count  avg_rating  median_rating  \
5     1890                No Language          187        4.22           4.80   
6     1890                    Unknown           43        2.30           2.00   
2     1890                     German            7        4.34           4.30   
12    1900                No Language          710        3.47           4.10   
15    1900                    Unknown          105        1.24           0.00   
9     1900                     French           18        4.04           4.95   
8     1900                    English           10        2.05           1.50   
28    1910                No Language         7166        1.16           0.00   
36    1910                    Unknown         2099        1.14           0.00   
18    1910                    English          155        1.87           0.00   
19    1910                     French    

In [8]:
# Cell 8: Analysis 6 - Highest Rated Genre by Country
print("\n" + "="*80)
print("ANALYSIS 6: HIGHEST RATED GENRE BY COUNTRY")
print("="*80)

start_time = time.time()
start_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
start_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

# Prepare data
countries = df.copy()
countries['production_countries'] = countries['production_countries'].str.split(', ')
countries = countries.explode('production_countries')
countries = countries[countries['production_countries'].notna() & (countries['production_countries'] != '')]

countries['genres'] = countries['genres'].str.split(', ')
countries = countries.explode('genres')
countries = countries[countries['genres'].notna() & (countries['genres'] != '')]

# Group by
countryGenre = countries.groupby(['production_countries', 'genres']).agg(
    avg_rating=('vote_average', 'mean'),
    movie_count=('id', 'count')
).round(2).reset_index()

countryGenre = countryGenre[countryGenre['movie_count'] >= 2]
countryGenre = countryGenre.sort_values(['production_countries', 'avg_rating'], ascending=[True, False])

# Get top genre per country
print("Top genre by country (first 20):")
countries_printed = 0
for country in sorted(countryGenre['production_countries'].unique())[:20]:
    country_data = countryGenre[countryGenre['production_countries'] == country]
    if len(country_data) > 0:
        top = country_data.iloc[0]
        print(f"{country}: {top['genres']} - {top['avg_rating']} avg rating ({top['movie_count']} movies)")
        countries_printed += 1

end_time = time.time()
end_memory = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
end_cpu = psutil.Process(os.getpid()).cpu_percent(interval=None)

execution_time = end_time - start_time
memory_delta = end_memory - start_memory
rows_processed = len(countryGenre)
rows_per_second = rows_processed / execution_time if execution_time > 0 else 0

# Store metrics for summary
analysis6_time = execution_time
analysis6_memory = memory_delta

print(f"\nBENCHMARKS:")
print(f"Time: {execution_time:.2f} seconds")
print(f"Memory delta: {memory_delta:.2f} MB")
print(f"Memory total: {end_memory:.2f} MB")
print(f"CPU: {end_cpu:.1f}%")
print(f"Genre-country pairs: {rows_processed:,}")
print(f"Pairs/second: {rows_per_second:,.0f}")
print(f"Countries: {len(countryGenre['production_countries'].unique())}")
print(f"Avg rating range: {countryGenre['avg_rating'].min():.2f} - {countryGenre['avg_rating'].max():.2f}")
print(f"Total genre-country pairs: {len(countryGenre):,}")
print("="*80)


ANALYSIS 6: HIGHEST RATED GENRE BY COUNTRY
Top genre by country (first 20):
Afghanistan: Documentary - 4.24 avg rating (34 movies)
Albania: Horror - 5.22 avg rating (8 movies)
Algeria: Adventure - 8.79 avg rating (10 movies)
American Samoa: Comedy - 1.38 avg rating (4 movies)
Andorra: War - 7.4 avg rating (2 movies)
Angola: Comedy - 7.05 avg rating (2 movies)
Antarctica: Adventure - 7.25 avg rating (2 movies)
Antigua And Barbuda: Documentary - 3.0 avg rating (2 movies)
Argentina: Adventure - 4.28 avg rating (170 movies)
Armenia: History - 6.54 avg rating (11 movies)
Aruba: Thriller - 5.2 avg rating (3 movies)
Australia: War - 4.76 avg rating (102 movies)
Austria: Adventure - 5.11 avg rating (86 movies)
Azerbaijan: Mystery - 2.88 avg rating (20 movies)
Bahamas: Adventure - 5.68 avg rating (4 movies)
Bahrain: Thriller - 5.25 avg rating (4 movies)
Bangladesh: Science Fiction - 3.6 avg rating (5 movies)
Barbados: Romance - 6.25 avg rating (2 movies)
Belarus: Animation - 4.6 avg rating (25

In [9]:
# Cell 9: Performance Summary (Auto-updating)
print("\n" + "="*80)
print("PERFORMANCE SUMMARY")
print("="*80)

# Collect results from all analyses
results = [
    {"Analysis": "Data Loading", "Time (s)": load_time if 'load_time' in locals() else 0, 
     "Memory (MB)": load_memory if 'load_memory' in locals() else 0},
    
    {"Analysis": "Genres by Country & Year", "Time (s)": analysis1_time if 'analysis1_time' in locals() else 0, 
     "Memory (MB)": analysis1_memory if 'analysis1_memory' in locals() else 0},
    
    {"Analysis": "Genres by Decade & Budget", "Time (s)": analysis2_time if 'analysis2_time' in locals() else 0, 
     "Memory (MB)": analysis2_memory if 'analysis2_memory' in locals() else 0},
    
    {"Analysis": "Genre by Company", "Time (s)": analysis3_time if 'analysis3_time' in locals() else 0, 
     "Memory (MB)": analysis3_memory if 'analysis3_memory' in locals() else 0},
    
    {"Analysis": "Multi vs Single Genre", "Time (s)": analysis4_time if 'analysis4_time' in locals() else 0, 
     "Memory (MB)": analysis4_memory if 'analysis4_memory' in locals() else 0},
    
    {"Analysis": "Language by Decade", "Time (s)": analysis5_time if 'analysis5_time' in locals() else 0, 
     "Memory (MB)": analysis5_memory if 'analysis5_memory' in locals() else 0},
    
    {"Analysis": "Highest Rated by Country", "Time (s)": analysis6_time if 'analysis6_time' in locals() else 0, 
     "Memory (MB)": analysis6_memory if 'analysis6_memory' in locals() else 0}
]

# Create DataFrame for better display
summary_df = pd.DataFrame(results)

# Calculate totals
total_time = summary_df["Time (s)"].sum()
total_memory = summary_df["Memory (MB)"].sum()

print("\n" + "-"*60)
print(f"{'Analysis':<35} {'Time (s)':<12} {'Memory (MB)':<12}")
print("-"*60)

for _, row in summary_df.iterrows():
    print(f"{row['Analysis']:<35} {row['Time (s)']:<12.2f} {row['Memory (MB)']:<12.2f}")

print("-"*60)
print(f"{'TOTAL':<35} {total_time:<12.2f} {total_memory:<12.2f}")
print("-"*60)

# Additional insights
print("\n" + "="*60)
print("INSIGHTS")
print("="*60)
if total_time > 0:
    max_time_idx = summary_df['Time (s)'].idxmax()
    max_memory_idx = summary_df['Memory (MB)'].idxmax()
    print(f"• Most time-consuming: {summary_df.loc[max_time_idx, 'Analysis']} ({summary_df['Time (s)'].max():.2f}s)")
    print(f"• Most memory-intensive: {summary_df.loc[max_memory_idx, 'Analysis']} ({summary_df['Memory (MB)'].max():.2f}MB)")
    print(f"• Average time per analysis: {total_time/len(results):.2f}s")
print(f"• Total dataset size: {len(df):,} rows" if 'df' in locals() else "• Total dataset size: N/A")
print(f"• Final memory usage: {psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024):.2f} MB")
print("="*80)


PERFORMANCE SUMMARY

------------------------------------------------------------
Analysis                            Time (s)     Memory (MB) 
------------------------------------------------------------
Data Loading                        9.07         2265.64     
Genres by Country & Year            0.49         174.17      
Genres by Decade & Budget           0.07         23.48       
Genre by Company                    7.87         -371.88     
Multi vs Single Genre               0.44         -57.70      
Language by Decade                  0.04         14.28       
Highest Rated by Country            4.98         983.17      
------------------------------------------------------------
TOTAL                               22.96        3031.17     
------------------------------------------------------------

INSIGHTS
• Most time-consuming: Data Loading (9.07s)
• Most memory-intensive: Data Loading (2265.64MB)
• Average time per analysis: 3.28s
• Total dataset size: 590,202 rows
• 

## MapReduce Implementation - for Genre by Company and for Highest rated by country 

In [11]:
# ============================================================
# MAPREDUCE - GENRE BY COUNTRY & YEAR
# ============================================================

from pyspark.sql import SparkSession
import time
import pandas as pd

# Spark setup
try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("GenreCountryYear_MapReduce") \
        .master("local[*]") \
        .getOrCreate()

sc = spark.sparkContext

start_time = time.time()

# Create year from release_date
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year

# Prepare data
df_mr = df[["genres", "production_countries", "year"]].copy()
df_mr["genres"] = df_mr["genres"].fillna("").astype(str).str.split(",")
df_mr["production_countries"] = df_mr["production_countries"].fillna("").astype(str).str.split(",")

records = df_mr.to_dict("records")
rdd = sc.parallelize(records)

result = (
    rdd.flatMap(lambda row: [
        ((genre.strip(), country.strip(), row["year"]), 1)
        for genre in row["genres"]
        for country in row["production_countries"]
        if genre.strip() != "" and country.strip() != "" and pd.notna(row["year"])
    ])
    .reduceByKey(lambda a, b: a + b)
    .map(lambda x: (x[0][0], x[0][1], int(x[0][2]), x[1]))
)

df_q1_mr = pd.DataFrame(
    result.collect(),
    columns=["genre", "country", "year", "count"]
).sort_values("count", ascending=False)

end_time = time.time()

print("Top results:")
print(df_q1_mr.head(10))
print(f"\nTime: {end_time - start_time:.2f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 12:55:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/19 12:55:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/19 12:55:43 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/19 12:55:43 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/19 12:55:46 WARN TaskSetManager: Stage 0 contains a task of very large size (2664 KiB). The maximum recommended task size is 1000 KiB.


Top results:
             genre        country  year  count
55344        Drama        Unknown  2024   3793
34979        Drama        Unknown  2025   3599
20305        Drama        Unknown  2023   2980
21047       Comedy        Unknown  2025   2366
57702       Comedy        Unknown  2024   2346
57379        Drama        Unknown  2022   2281
64890       Comedy        Unknown  2023   1894
18815        Drama        Unknown  2021   1856
47012        Drama  United States  2024   1677
42547  Documentary        Unknown  2024   1671

Time: 3.83s


In [12]:
# ============================================================
# MAPREDUCE - HIGHEST RATED BY COUNTRY
# ============================================================

from pyspark.sql import SparkSession
import time
import pandas as pd

# Spark setup
try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("HighestRatedCountry_MapReduce") \
        .master("local[*]") \
        .getOrCreate()

sc = spark.sparkContext

start_time = time.time()

# Prepare data
df_mr = df[["title", "production_countries", "vote_average"]].copy()
df_mr["production_countries"] = df_mr["production_countries"].fillna("").astype(str).str.split(",")

records = df_mr.to_dict("records")
rdd = sc.parallelize(records)

result = (
    rdd.flatMap(lambda row: [
        (country.strip(), (row["vote_average"], row["title"]))
        for country in row["production_countries"]
        if country.strip() != "" and pd.notna(row["vote_average"])
    ])
    .reduceByKey(lambda a, b: a if a[0] >= b[0] else b)
)

df_q6_mr = pd.DataFrame(result.collect(), columns=["country", "top_movie"])
df_q6_mr[["rating", "title"]] = pd.DataFrame(df_q6_mr["top_movie"].tolist(), index=df_q6_mr.index)
df_q6_mr = df_q6_mr.drop(columns=["top_movie"]).sort_values("rating", ascending=False)

end_time = time.time()

print("Top results:")
print(df_q6_mr.head(10))
print(f"\nTime: {end_time - start_time:.2f}s")

26/03/19 12:55:49 WARN TaskSetManager: Stage 2 contains a task of very large size (2855 KiB). The maximum recommended task size is 1000 KiB.


Top results:
           country  rating                         title
0           Sweden    10.0   Imperiet: Här föll Imperiet
142       Ethiopia    10.0                   Gazetegnawa
128  Guinea-Bissau    10.0                       Hepicat
129         Malawi    10.0                        Malawi
135        Germany    10.0                        Collin
136          Italy    10.0  Eros Ramazzotti: Stilelibero
137        Hungary    10.0             Tales of Budapest
138       Slovakia    10.0                    66 Seasons
139      Singapore    10.0                Mat Raja Kapor
140       Botswana    10.0                  Matabeleland

Time: 1.63s
